In [198]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/saurabhshahane/fake-news-classification/WELFake_Dataset.csv
/kaggle/input/datasets/firestorm713/newsdataset/News_Dataset/True.csv
/kaggle/input/datasets/firestorm713/newsdataset/News_Dataset/Fake.csv


In [199]:
import numpy as np 
import pandas as pd 
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk.corpus import stopwords 
from collections import Counter
import string
import re
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [15]:
import os
news_ds_path = '/kaggle/input/datasets/firestorm713/newsdataset/News_Dataset'
fake_ds_path = os.path.join(news_ds_path, 'Fake.csv')
true_ds_path = os.path.join(news_ds_path, 'True.csv')

In [16]:
fake_ds = pd.read_csv(fake_ds_path)
true_ds = pd.read_csv(true_ds_path)

In [5]:
print(fake_ds.head(1))
print(fake_ds.columns)

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   

                date  
0  December 31, 2017  
Index(['title', 'text', 'subject', 'date'], dtype='object')


## Labels Distribution

In [26]:
print('Fake news %: ', len(fake_ds)/(len(fake_ds)+len(true_ds)) * 100)
print('True news %: ', len(true_ds)/(len(fake_ds)+len(true_ds)) * 100)

Fake news %:  52.29854336496058
True news %:  47.70145663503943


In [27]:
true_ds['subject'].value_counts()

subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64

In [28]:
fake_ds['subject'].value_counts()

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

### Subject names differ, this can cause data leakage for model, so drop this column

In [6]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [200]:
is_cuda = torch.cuda.is_available()
if is_cuda:
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU not available, CPU used")

GPU is available


In [ ]:
def padding_(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, sentence in enumerate(sentences):
        if len(sentence) > 0:
            features[ii, -len(sentence):] = np.array(sentence)[:seq_len]
    return features

In [ ]:
def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)
    return s

def tokenize(x_train,y_train,x_val,y_val):
    word_list = []

    stop_words = set(stopwords.words('english')) 
    for sent in x_train:
        for word in sent.lower().split():
            word = preprocess_string(word)
            if word not in stop_words and word != '':
                word_list.append(word)

    corpus = Counter(word_list)
    # sorting on the basis of most common words
    corpus_ = sorted(corpus,key=corpus.get,reverse=True)[:1000]
    # creating a dict
    onehot_dict = {w:i+1 for i,w in enumerate(corpus_)}

    # tokenize
    final_list_train,final_list_test = [],[]
    for sent in x_train:
            final_list_train.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split() 
                                     if preprocess_string(word) in onehot_dict.keys()])
    for sent in x_val:
            final_list_test.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split() 
                                    if preprocess_string(word) in onehot_dict.keys()])

    encoded_train = [1 if label =='positive' else 0 for label in y_train]  
    encoded_test = [1 if label =='positive' else 0 for label in y_val] 
    final_list_train = padding_(final_list_train, 500)
    final_list_test = padding_(final_list_test, 500)
    return np.array(final_list_train), np.array(encoded_train),np.array(final_list_test), np.array(encoded_test),onehot_dict

In [258]:
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# def preprocess_text(text):
#     # 1. Neglecting casing (Lowercasing)
#     text = str(text).lower()
#     # 2. Removing punctuation and non-alphabetic characters
#     text = re.sub(r'[^a-z\s]', '', text)
#     # 3. Removing stop words
#     words = text.split()
#     cleaned_words = [w for w in words if w not in stop_words]
#     return " ".join(cleaned_words)

def preprocess_text(text):
    # 1. Lowercase
    text = str(text).lower()
    
    # 2. Rescue hyphenated words and slashes by turning them into spaces
    # "health-care" -> "health care", "9/11" -> "9 11"
    text = re.sub(r'[-/]', ' ', text)
    
    # 3. Rescue possessives
    # "obama's" -> "obama", "florida's" -> "florida"
    text = re.sub(r"'s\b", '', text)
    
    # 4. Remove all remaining punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 5. Remove stop words
    words = text.split()
    cleaned_words = [w for w in words if w not in stop_words]
    
    return " ".join(cleaned_words)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [201]:
class GloveTokenizer:
    def __init__(self, texts, max_vocab=10000):
        # Build vocab from the preprocessed texts
        all_words = " ".join(texts).split()
        word_counts = Counter(all_words)
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        for word, _ in word_counts.most_common(max_vocab):
            self.word_index[word] = len(self.word_index)

    @property
    def vocab_size(self):
        return len(self.word_index)
            
    def __call__(self, text, padding=None, truncation=True, max_length=100, return_tensors='pt'):
        # Handle both single strings and lists of strings
        if isinstance(text, str): text = [text]
        
        batch_ids = []
        for s in text:
            ids = [self.word_index.get(w, 1) for w in s.split()][:max_length]
            # Padding
            ids += [0] * (max_length - len(ids))
            batch_ids.append(ids)
            
        return {'input_ids': torch.tensor(batch_ids)}

In [202]:
class FakeNewsDataset(Dataset):
    def __init__(self, raw_texts, labels=None, tokenizer=None, max_seq_len=500):
        if tokenizer is None:
            raise ValueError("Got None tokenizer")
        self.encodings = tokenizer(
            text = raw_texts,
            padding = 'max_length',
            truncation = True,
            max_length = max_seq_len,
            return_tensors = 'pt'
        )
        # self.labels = torch.tensor(labels) if labels is not None else None
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])


    def __getitem__(self, idx):
        item_tokens = self.encodings['input_ids'][idx]
        label = self.labels[idx] if self.labels is not None else None
        return item_tokens, label
        

In [39]:
from transformers import AutoTokenizer

# We use RoBERTa's tokenizer because it is a highly optimized Byte-Level BPE tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

vocab_size = tokenizer.vocab_size
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 50265


In [10]:
# test tokenizer output
tokenizer('Training Set:   34346 texts')['input_ids']

[0, 44466, 8504, 35, 1437, 1437, 2631, 34088, 14301, 2]

In [11]:
def prepare_text_for_embedding(text_list, max_seq_length=500, framework='pt'):
    encoded_data = tokenizer(
        text_list,
        padding='max_length',     
        truncation=True,         
        max_length=max_seq_length,
        return_tensors=framework  
    )
    return encoded_data['input_ids']

In [53]:
import re

def clean_leakage(text):
    # 1. First, remove the common "Agency Tag" preamble entirely 
    # (e.g., "WASHINGTON (Reuters) - ")
    tag_pattern = r'^.*?\(\s*reuters\s*\)\s*[-—:]\s*'
    text = re.sub(tag_pattern, '', text, flags=re.IGNORECASE)

    # 2. Now, remove the word "reuters" EVERYWHERE else in the text
    # \b ensures we match the whole word only
    global_pattern = r'\breuters\b'
    text = re.sub(global_pattern, '', text, flags=re.IGNORECASE)

    # 3. Clean up any double spaces left behind by the removal
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [54]:
def get_texts_and_labels(fake_df, true_df):
    def concat_text_features(df):
        return df['text'].fillna('')
        # return 'Title: ' + df['title'].fillna('') + '. ' + df['text'].fillna('')
        # return 'Subject: ' + df['subject'] + ' Title: ' + df['title'] + '. ' + df['text']
    fake_texts = concat_text_features(fake_df).tolist()
    true_texts = concat_text_features(true_df).tolist()
    texts = fake_texts + true_texts
    labels = torch.concat((torch.zeros(len(fake_texts), dtype=int), 
                          torch.ones(len(true_texts), dtype=int)))
    texts_cleaned = [clean_leakage(t) for t in texts]
    return texts_cleaned, labels

In [55]:
texts, labels = get_texts_and_labels(fake_ds, true_ds)
print(texts[0][:150])
print(labels[0])

Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and th
tensor(0)


In [56]:
from sklearn.model_selection import train_test_split

texts_train_val, texts_test, labels_train_val, labels_test = train_test_split(
    texts, 
    labels, 
    test_size=0.10, 
    random_state=42,     
    stratify=labels     
)

texts_train, texts_val, labels_train, labels_val = train_test_split(
    texts_train_val, 
    labels_train_val, 
    test_size=0.15,    
    random_state=42, 
    stratify=labels_train_val
)

print(f"Training Set:   {len(texts_train)} texts")
print(f"Validation Set: {len(texts_val)} texts")
print(f"Test Set:       {len(texts_test)} texts")

train_ds = FakeNewsDataset(texts_train, labels_train, tokenizer=tokenizer)
val_ds = FakeNewsDataset(texts_val, labels_val, tokenizer=tokenizer)
test_ds = FakeNewsDataset(texts_test, labels_test, tokenizer=tokenizer)

batch_size = 16

train_loader = DataLoader(train_ds, shuffle=True, batch_size=batch_size)
val_loader = DataLoader(val_ds, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_ds, shuffle=True, batch_size=batch_size)

Training Set:   34346 texts
Validation Set: 6062 texts
Test Set:       4490 texts


In [203]:
class FakeNewsLstm(nn.Module):
    def __init__(self, no_layers, vocab_size, hidden_dim, embedding_dim, output_dim=1, drop_prob=0.35):
        super(FakeNewsLstm, self).__init__()

        self.no_layers = no_layers
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Can add bidirectional=True for better context
        self.lstm = nn.LSTM(input_size=embedding_dim, 
                            hidden_size=self.hidden_dim,
                            num_layers=no_layers, 
                            batch_first=True,
                            dropout=drop_prob if no_layers > 1 else 0)

        self.dropout = nn.Dropout(drop_prob)
        self.fc1 = nn.Linear(self.hidden_dim, 32)
        self.act = nn.ReLU()
        self.fc = nn.Linear(32, output_dim)

    # def forward(self, x, hidden):
    #     # 1. Embeddings: (Batch, Seq_Len, Embed_Dim)
    #     embeds = self.embedding(x)
        
    #     # 2. LSTM: 
    #     # lstm_out shape: (Batch, Seq_Len, Hidden_Dim)
    #     lstm_out, hidden = self.lstm(embeds, hidden)

    #     # 3. Get the last time step's output
    #     # Instead of flattening everything, we just grab the last word's output
    #     # shape becomes: (Batch, Hidden_Dim)
    #     last_time_step = lstm_out[:, -1, :]

    #     out = self.dropout(last_time_step)
    #     logits = self.fc(out) # Returning raw logits

    #     return logits, hidden

    def forward(self, x, hidden):
        # x shape: [batch, seq_len]
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)
        
        # NEW LOGIC: Find the last non-zero token for every sentence in the batch
        # This assumes 0 is your <PAD> token
        lengths = (x != 0).sum(dim=1).cpu() 
        
        # Extract the output at the last word index for each sample
        # Instead of just [:, -1, :], we do:
        idx = (lengths - 1).view(-1, 1).expand(len(lengths), lstm_out.size(2)).unsqueeze(1)
        idx = idx.to(device) # move to GPU
        last_word_out = lstm_out.gather(1, idx).squeeze(1)
        x_out = self.dropout(last_word_out)
        x_out = self.act(self.fc1(x_out))
        # Optional but recommended: Second Dropout to protect the new Dense layer
        x_out = self.dropout(x_out)
        x_out = self.fc(x_out)
        return x_out, hidden

    def init_hidden(self, batch_size, device):
        h0 = torch.zeros((self.no_layers, batch_size, self.hidden_dim)).to(device)
        c0 = torch.zeros((self.no_layers, batch_size, self.hidden_dim)).to(device)
        return (h0, c0)

In [85]:
NUM_LAYERS = 1
VOCAB_SIZE = tokenizer.vocab_size
HIDDEN_DIM = 16
EMBEDDING_DIM = 32
OUTPUT_DIM = 1
DROP_PROB = 0.35
model = FakeNewsLstm(no_layers=NUM_LAYERS, vocab_size=VOCAB_SIZE,
                    hidden_dim=HIDDEN_DIM, embedding_dim=EMBEDDING_DIM,
                    output_dim=OUTPUT_DIM, drop_prob=DROP_PROB).to(device)

In [204]:
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

In [87]:
optimizer = Adam(model.parameters(), lr=1e-3)
sched = StepLR(optimizer, step_size=10, gamma=0.1)
criterion = nn.BCEWithLogitsLoss()

In [205]:
from sklearn.metrics import accuracy_score, f1_score

In [206]:
def acc(out, labels):
    out = torch.sigmoid(out)
    preds = out >= 0.5
    return accuracy_score(y_true=labels, y_pred=preds, normalize=False)

In [207]:
def acc_multi(out, labels):
    out = torch.softmax(out, dim=1)
    preds = torch.argmax(out, dim=1)
    return accuracy_score(y_true=labels, y_pred=preds, normalize=False)

In [208]:
def train_epochs(model, optimizer, criterion, scheduler=None, epochs=5, binary=False, savepath='top_model.pt'):
    clip = 5
    # valid_loss_min = np.inf
    val_acc_max = 0.62
    epoch_tr_loss,epoch_vl_loss = [],[]
    epoch_tr_acc,epoch_vl_acc = [],[]
    
    for epoch in range(epochs):
        train_losses = []
        train_acc = 0.0
        model.train()
        # initialize hidden state 
        # h = model.init_hidden(batch_size)

        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)  
            h = model.init_hidden(inputs.size(0), device)
            # Creating new variables for the hidden state, otherwise
            # we'd backprop through the entire training history
            # h = tuple([each.detach() for each in h])
    
            optimizer.zero_grad()
            output,h = model(inputs,h)
    
            # calculate the loss and perform backprop
            if isinstance(criterion, nn.CrossEntropyLoss):
                loss = criterion(output.squeeze(), labels.long())
            else:
                loss = criterion(output.squeeze(), labels.float())
            loss.backward()
            # train_bar.write(f'Gradient: {model.lstm.weight_ih_l0.grad.abs().sum().item()}')
            train_losses.append(loss.item())
            # calculating accuracy
            if binary:
                accuracy = acc(output.detach().cpu(),labels.detach().cpu())
            else:
                accuracy = acc_multi(output.detach().cpu(),labels.detach().cpu())
            # accuracy = acc(output.detach().cpu(),labels.detach().cpu())
            train_acc += accuracy
            #`clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
            # nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            train_bar.set_postfix({
                "train_loss": f"{loss.item():.4f}"
            })

        if scheduler:
            scheduler.step()
    
        # val_h = model.init_hidden(batch_size)
        val_losses = []
        val_acc = 0.0
        model.eval()

        val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Valid]')

        i = 0
        with torch.no_grad():
            for inputs, labels in val_bar:
                val_h = model.init_hidden(inputs.size(0), device)

                # val_h = tuple([each.detach() for each in val_h])
    
                inputs, labels = inputs.to(device), labels.to(device)
    
                output, val_h = model(inputs, val_h)
                # if i==0:
                #     print(torch.sigmoid(output))
                #     i=7
                if isinstance(criterion, nn.CrossEntropyLoss):
                    val_loss = criterion(output.squeeze(), labels.long())
                else:
                    val_loss = criterion(output.squeeze(), labels.float())
                # val_loss = criterion(output.squeeze(), labels.float())
    
                val_losses.append(val_loss.item())

                if binary:
                    accuracy = acc(output.detach().cpu(),labels.detach().cpu())
                else:
                    accuracy = acc_multi(output.detach().cpu(),labels.detach().cpu())

                val_acc += accuracy
                val_bar.set_postfix({
                    "val_loss": f"{val_loss.item():.4f}",
                })
    
        epoch_train_loss = np.mean(train_losses)
        epoch_val_loss = np.mean(val_losses)
        epoch_train_acc = train_acc/len(train_loader.dataset)
        epoch_val_acc = val_acc/len(val_loader.dataset)

        # Save model if validation loss decreased
        if epoch_val_acc > val_acc_max:
            # print(f'Validation loss decreased ({valid_loss_min:.6f} --> {epoch_val_loss:.6f}). Saving model...')
            torch.save(model.state_dict(), savepath)
            val_acc_max = epoch_val_acc
        
        print(f'Train loss: {round(epoch_train_loss,3)}, Train acc: {round(epoch_train_acc,3)}')
        print(f'Val loss: {round(epoch_val_loss,3)}, Val acc: {round(epoch_val_acc, 3)}')
        epoch_tr_loss.append(epoch_train_loss)
        epoch_vl_loss.append(epoch_val_loss)
        epoch_tr_acc.append(epoch_train_acc)
        epoch_vl_acc.append(epoch_val_acc)
    return epoch_tr_loss, epoch_vl_loss, epoch_tr_acc, epoch_vl_acc

In [94]:
num_epochs = 2
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs)

Epoch 1/2 [Valid]: 100%|██████████| 379/379 [00:01<00:00, 245.32it/s, val_loss=0.0029]


Train loss: 0.375, Train acc: 0.79
Val loss: 0.039, Val acc: 0.993


Epoch 2/2 [Valid]: 100%|██████████| 379/379 [00:01<00:00, 243.24it/s, val_loss=0.0048]


Train loss: 0.071, Train acc: 0.982
Val loss: 0.033, Val acc: 0.994


In [ ]:
import matplotlib.pyplot as plt
def display_evolution(train_losses, val_losses, train_acc_list, val_acc_list):
    

In [91]:
batch_size = 32
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True)

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 335.64it/s, val_loss=1.0772]


Train loss: 0.342, Train acc: 0.861
Val loss: 0.969, Val acc: 0.57


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 328.52it/s, val_loss=0.4120]


Train loss: 0.334, Train acc: 0.864
Val loss: 0.994, Val acc: 0.574


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 337.66it/s, val_loss=2.2521]


Train loss: 0.322, Train acc: 0.872
Val loss: 1.047, Val acc: 0.576


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 334.96it/s, val_loss=0.2507]


Train loss: 0.314, Train acc: 0.874
Val loss: 1.008, Val acc: 0.576


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 331.59it/s, val_loss=1.9736]

Train loss: 0.312, Train acc: 0.877
Val loss: 1.067, Val acc: 0.575


In [77]:
batch_size = 32
num_epochs = 3
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True)

Epoch 1/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 283.07it/s, val_loss=0.7891]


Train loss: 0.05, Train acc: 0.985
Val loss: 1.813, Val acc: 0.593


Epoch 2/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 277.09it/s, val_loss=3.2926]


Train loss: 0.052, Train acc: 0.984
Val loss: 1.873, Val acc: 0.57


Epoch 3/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 293.11it/s, val_loss=1.6989]

Train loss: 0.028, Train acc: 0.994
Val loss: 1.903, Val acc: 0.576


In [71]:
eval_model(model, test_loader)

Epoch 1/1 [Valid]: 100%|██████████| 40/40 [00:00<00:00, 254.62it/s, val_loss=2.5782]

Val acc:  0.5816890292028414


In [102]:



texts_train, labels_train = get_texts_and_labels(liar_train)
texts_val, labels_val = get_texts_and_labels(liar_val)
texts_test, labels_test = get_texts_and_labels(liar_test)

print("4. Tokenizing and Evaluating...")
# Re-use your exact FakeNewsDataset class
unseen_dataset = FakeNewsDataset(
    raw_texts=liar_texts, 
    labels=liar_labels, 
    tokenizer=tokenizer, 
    max_seq_len=500 
)

unseen_loader = DataLoader(unseen_dataset, batch_size=32, shuffle=False)

# Evaluate!
model.eval()
test_losses = []
test_correct = 0

with torch.no_grad():
    for inputs, labels in unseen_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        test_h = model.init_hidden(inputs.size(0), device)
        
        output, test_h = model(inputs, test_h)
        
        accuracy = acc(output.detach().cpu(), labels.detach().cpu())
        test_correct += accuracy

final_unseen_acc = test_correct / len(unseen_loader.dataset)
print(f"--------------------------------------------------")
print(f"Accuracy on completely unseen LIAR dataset: {final_unseen_acc:.4f}")
print(f"--------------------------------------------------")

1. Downloading raw LIAR dataset from UC Santa Barbara...
2. Extracting files...
3. Loading and processing data with Pandas...
Successfully loaded 1267 unseen statements.
4. Tokenizing and Evaluating...
--------------------------------------------------
Accuracy on completely unseen LIAR dataset: 0.4365
--------------------------------------------------


## The model learnt easy semantic features on ISOT dataset, 
## however, it struggles with more complicated datasets
## Let's try another dataset

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/saurabhshahane/fake-news-classification


In [6]:
df = pd.read_csv('/kaggle/input/datasets/saurabhshahane/fake-news-classification/WELFake_Dataset.csv')
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
# 0 = fake, 1 = true
df['label'].value_counts()

label
1    37106
0    35028
Name: count, dtype: int64

In [8]:
df.isna().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [10]:
df = df.dropna(how='all').fillna('')

In [11]:
def get_texts_and_labels(df):
    def concat_text_features(df):
        return 'Title: ' + df['title'].fillna('') + '. ' + df['text'].fillna('')
        # return 'Subject: ' + df['subject'] + ' Title: ' + df['title'] + '. ' + df['text']
    
    texts = concat_text_features(df).tolist()
    labels = torch.tensor(df['label'].values, dtype=int)
    return texts, labels

In [12]:
texts, labels = get_texts_and_labels(df)
print(texts[0][:100])
print(labels[0])

Title: LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLives
tensor(1)


In [22]:
from sklearn.model_selection import train_test_split

def prepare_loaders(texts, labels):
    texts_train_val, texts_test, labels_train_val, labels_test = train_test_split(
        texts, 
        labels, 
        test_size=0.10, 
        random_state=42,     
        stratify=labels     
    )
    
    texts_train, texts_val, labels_train, labels_val = train_test_split(
        texts_train_val, 
        labels_train_val, 
        test_size=0.15,    
        random_state=42, 
        stratify=labels_train_val
    )
    
    print(f"Training Set:   {len(texts_train)} texts")
    print(f"Validation Set: {len(texts_val)} texts")
    print(f"Test Set:       {len(texts_test)} texts")
    
    train_ds = FakeNewsDataset(texts_train, labels_train, tokenizer=tokenizer)
    val_ds = FakeNewsDataset(texts_val, labels_val, tokenizer=tokenizer)
    test_ds = FakeNewsDataset(texts_test, labels_test, tokenizer=tokenizer)
    
    batch_size = 16
    
    train_loader = DataLoader(train_ds, shuffle=True, batch_size=batch_size)
    val_loader = DataLoader(val_ds, shuffle=True, batch_size=batch_size)
    test_loader = DataLoader(test_ds, shuffle=True, batch_size=batch_size)
    return train_loader, val_loader, test_loader

In [23]:
train_loader, val_loader, test_loader = prepare_loaders(texts, labels)

Training Set:   55182 texts
Validation Set: 9738 texts
Test Set:       7214 texts


## Try predictions with prev model

In [209]:
def eval_model(model, val_loader):
    val_losses = []
    val_acc = 0.0
    model.eval()
    val_bar = tqdm(val_loader, desc=f'Epoch {1}/{1} [Valid]')
    
    with torch.no_grad():
        for inputs, labels in val_bar:
            val_h = model.init_hidden(inputs.size(0), device)
            inputs, labels = inputs.to(device), labels.to(device)

            output, val_h = model(inputs, val_h)
            val_loss = criterion(output.squeeze(), labels.float())

            val_losses.append(val_loss.item())

            accuracy = acc(output.detach().cpu(),labels.detach().cpu())
            val_acc += accuracy
            val_bar.set_postfix({
                "val_loss": f"{val_loss.item():.4f}",
            })

    epoch_val_loss = np.mean(val_losses)
    epoch_val_acc = val_acc/len(val_loader.dataset)
    print('Val acc: ', epoch_val_acc)

In [35]:
model = model.to(device)
model.load_state_dict(torch.load('/kaggle/working/lstm_best_model_v2_cpu.pt', weights_only=True))

<All keys matched successfully>

In [39]:
eval_model(model, val_loader)

Epoch 1/1 [Valid]: 100%|██████████| 379/379 [00:15<00:00, 25.16it/s, val_loss=6.8443]

Val acc:  0.9986803035301881


## Model trained on WELFake dataset performs well on ISOT dataset also!

## Fast check of random forest model

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Vectorization: Convert text to numbers
# We limit max_features to 5000 to keep the forest fast and avoid memory issues
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

print("Vectorizing text...")

# Apply it to your existing list
# texts_train = [clean_leakage(t) for t in texts_train]
# texts_test = [clean_leakage(t) for t in texts_test]
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

# 2. Initialize Random Forest
# n_jobs=-1 uses all available CPU cores
# n_estimators=100 is a good balance between speed and accuracy
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

print("Training Random Forest (this might take a minute)...")
rf_model.fit(X_train, labels_train)

# 3. Predict and Evaluate
print("Evaluating...")
y_pred = rf_model.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(labels_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(labels_test, y_pred))

Vectorizing text...
Training Random Forest (this might take a minute)...
Evaluating...
Random Forest Accuracy: 0.9837

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      2348
           1       0.98      0.99      0.98      2142

    accuracy                           0.98      4490
   macro avg       0.98      0.98      0.98      4490
weighted avg       0.98      0.98      0.98      4490



In [58]:
# Get feature names from the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Get importances from the RF model
importances = rf_model.feature_importances_

# Sort and show the top 10
indices = np.argsort(importances)[-10:]
print("Top 10 most 'important' words for the model:")
for i in reversed(indices):
    print(f"{feature_names[i]}: {importances[i]:.4f}")

Top 10 most 'important' words for the model:
said: 0.0625
featured: 0.0280
image: 0.0251
com: 0.0147
just: 0.0145
like: 0.0132
pic: 0.0120
watch: 0.0118
minister: 0.0113
wednesday: 0.0112


In [59]:
src_fake_text = '''GENEVA — The World Trade Organization (WTO) announced on Wednesday a preliminary framework for the Global Semiconductor Accord (GSA), a treaty designed to standardize silicon export quotas across four continents.

Speaking from the headquarters in Geneva, WTO Director-General Ngozi Okonjo-Iweala said the agreement would stabilize the volatile microchip market by 2027. "This framework represents a significant shift in how technological commodities are managed globally," she said during a press conference.

Under the terms of the draft, signatory nations would be required to maintain a minimum strategic reserve of industrial-grade neon, a key component in laser lithography. Analysts said the move is likely aimed at reducing the influence of independent brokerage firms in the Asia-Pacific region.'''
cleaned_test = heavy_clean_leakage(src_fake_text)

# 2. Vectorize
vec_test = vectorizer.transform([cleaned_test])

# 3. Predict Probability
# [Probability of Fake, Probability of True]
prob = rf_model.predict_proba(vec_test)

print(f"Model Prediction: {'TRUE' if prob[0][1] > 0.5 else 'FAKE'}")
print(f"Confidence: {prob[0][max(0,1)] * 100:.2f}%")

Model Prediction: TRUE
Confidence: 90.00%


## All models so far did not learn anything useful apart from syntax features (pro vs conversational style)

In [65]:
claim = '''GENEVA — The World Trade Organization (WTO) announced on Wednesday a preliminary framework for the Global Semiconductor Accord (GSA), a treaty designed to standardize silicon export quotas across four continents.'''
query = "World Trade Organization Global Semiconductor Accord"
evidence = retrieve_wikipedia_evidence(query, num_results=3)
# Resulting Evidence:
# Doc 1: "Ngozi Okonjo-Iweala is the Director-General of the World Trade Organization..."
# Doc 2: "The 14th WTO Ministerial Conference (MC14) took place in Cameroon in March 2026, focusing on e-commerce and dispute settlement..."
# Doc 3: "A semiconductor is a material which has an electrical conductivity value falling between that of a conductor and an insulator..."

# 2. NLI Verification Phase
pairs = [[doc, claim] for doc in evidence]
predictions = nli_model.predict(pairs)

print(predictions)
# Output Probabilities: [Contradiction, Entailment, Neutral]
# Doc 1: [0.05, 0.01, 0.94] -> NEUTRAL
# Doc 2: [0.35, 0.02, 0.63] -> NEUTRAL 
# Doc 3: [0.02, 0.01, 0.97] -> NEUTRAL

print("Final Verdict: UNVERIFIED (Likely Fake/Hallucination)")


🔍 Searching Wikipedia for: 'World Trade Organization Global Semiconductor Accord' (fetching up to 3 pages)...
✅ Successfully retrieved 3 Wikipedia summaries.
[[-2.151445  -3.2631612  5.7149973]
 [-1.586399  -2.1383166  3.8748133]
 [-1.3836058 -2.9084516  4.6692576]]
Final Verdict: UNVERIFIED (Likely Fake/Hallucination)


## Train on new Dataset

In [40]:
num_epochs = 2
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs)

Epoch 1/2 [Valid]: 100%|██████████| 609/609 [00:23<00:00, 26.00it/s, val_loss=0.0160]


Train loss: 0.401, Train acc: 0.777
Val loss: 0.078, Val acc: 0.976


Epoch 2/2 [Valid]: 100%|██████████| 609/609 [00:24<00:00, 25.01it/s, val_loss=0.0039]


Train loss: 0.041, Train acc: 0.987
Val loss: 0.027, Val acc: 0.991


## Test on Liar Dataset

In [41]:
import pandas as pd
import urllib.request
import zipfile
import os
import torch
from torch.utils.data import DataLoader

print("1. Downloading raw LIAR dataset from UC Santa Barbara...")
url = "http://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
urllib.request.urlretrieve(url, "liar_dataset.zip")

print("2. Extracting files...")
with zipfile.ZipFile("liar_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("liar_data")

print("3. Loading and processing data with Pandas...")
# Safest way to read the TSV: don't guess column names, just use indices
liar_test = pd.read_csv("liar_data/test.tsv", sep='\t', header=None, on_bad_lines='skip')

# In the LIAR dataset:
# Column 1 is the Label
# Column 2 is the Statement
# We cast to string, lowercase, and strip all hidden whitespace
liar_test['clean_label'] = liar_test[1].astype(str).str.lower().str.strip()
liar_test['statement'] = liar_test[2].astype(str)

# Map the 6-way labels to Binary
label_mapping = {
    'pants-fire': 0,
    'false': 0,
    'barely-true': 0,
    'half-true': 1,
    'mostly-true': 1,
    'true': 1
}

liar_test['binary_label'] = liar_test['clean_label'].map(label_mapping)

# Drop any rows that STILL failed the mapping (just in case)
liar_test = liar_test.dropna(subset=['binary_label'])

# Extract standard Python lists
liar_texts = liar_test['statement'].tolist()
liar_labels = liar_test['binary_label'].astype(int).tolist()

print(f"Successfully loaded {len(liar_texts)} unseen statements.")

# Crash prevention check:
if len(liar_texts) == 0:
    raise ValueError("CRITICAL: No data was loaded! The label mapping failed.")

# ---------------------------------------------------------
# 4. Push through your existing pipeline
# ---------------------------------------------------------
print("4. Tokenizing and Evaluating...")
# Re-use your exact FakeNewsDataset class
unseen_dataset = FakeNewsDataset(
    raw_texts=liar_texts, 
    labels=liar_labels, 
    tokenizer=tokenizer, 
    max_seq_len=500 
)

unseen_loader = DataLoader(unseen_dataset, batch_size=32, shuffle=False)

# Evaluate!
model.eval()
test_losses = []
test_correct = 0

with torch.no_grad():
    for inputs, labels in unseen_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        test_h = model.init_hidden(inputs.size(0), device)
        
        output, test_h = model(inputs, test_h)
        
        accuracy = acc(output.detach().cpu(), labels.detach().cpu())
        test_correct += accuracy

final_unseen_acc = test_correct / len(unseen_loader.dataset)
print(f"--------------------------------------------------")
print(f"Accuracy on completely unseen LIAR dataset: {final_unseen_acc:.4f}")
print(f"--------------------------------------------------")

1. Downloading raw LIAR dataset from UC Santa Barbara...
2. Extracting files...
3. Loading and processing data with Pandas...
Successfully loaded 1267 unseen statements.
4. Tokenizing and Evaluating...
--------------------------------------------------
Accuracy on completely unseen LIAR dataset: 0.4633
--------------------------------------------------


In [5]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=469177bdae7ee0d98353e0f64b9603bf15484158d24197152f29a2f1f27ce1d2
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


## Keyword based wiki search

In [3]:
import wikipedia

def retrieve_wikipedia_evidence(claim, num_results=3):
    print(f"Searching Wikipedia for: '{claim}'...")
    
    # 1. Search Wikipedia (Returns a list of page titles)
    # By default, Wikipedia ALREADY sorts these by its own relevance algorithm
    search_results = wikipedia.search(claim, results=num_results)
    
    if not search_results:
        return ["No relevant Wikipedia pages found."]
    
    evidence_chunks = []
    
    # 2. Fetch the actual text from those pages
    for title in search_results:
        try:
            # We use .summary() to just get the introduction. 
            # Downloading the whole page is usually too much text for an LLM to read efficiently.
            summary = wikipedia.summary(title, sentences=5)
            evidence_chunks.append(summary)
            
        except wikipedia.exceptions.DisambiguationError as e:
            # This happens if your search is too broad (e.g., searching "Apple")
            print(f"Skipping '{title}' (Disambiguation page)")
        except wikipedia.exceptions.PageError:
            # The page doesn't exist anymore
            print(f"Skipping '{title}' (Page not found)")
            
    return evidence_chunks

# Try it out:
claim = "The moon is made of green cheese."
evidence = retrieve_wikipedia_evidence(claim)
for i, text in enumerate(evidence):
    print(f"\n--- Result {i+1} ---")
    print(text)

Searching Wikipedia for: 'The moon is made of green cheese.'...

--- Result 1 ---
"The Moon is made of green cheese" is a statement referring to a fanciful belief that the Moon is composed of cheese. In its original formulation as a proverb and metaphor for credulity with roots in fable, this refers to the perception of a simpleton who sees a reflection of the Moon in water and mistakes it for a round cheese wheel. It is widespread as a folkloric motif among many of the world's cultures, and the notion has also found its way into children's folklore and modern popular culture.

The phrase "green cheese" in the common version of this proverb (sometimes "cream cheese" is used), may refer to a young, unripe cheese or to cheese with a greenish tint.
There was never an actual historical popular belief that the Moon is made of green cheese (cf.

--- Result 2 ---
Green cheese is a fresh cheese that has not thoroughly dried nor aged, which is white in color and usually round in shape. The Oxfo

## Search with re-ranker

In [4]:
from sentence_transformers import CrossEncoder

# Load a pre-trained re-ranker model specifically built for search relevance
# This model is tiny and runs very fast on a CPU
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

claim = "Does the moon actually consist of green cheese?"

# Let's pretend these 3 chunks came from our Wikipedia function
wiki_chunks = [
    "Cheese is a dairy product produced in wide ranges of flavors, textures, and forms by coagulation of the milk protein casein.",
    "The Moon is Earth's only natural satellite. It is a planetary-mass object with a differentiated rocky body.",
    "The Moon is made of green cheese is a proverb and statement that refers to a fanciful belief that the Moon is composed of cheese."
]

# 1. Pair the claim with every single chunk of evidence
pairs = [[claim, chunk] for chunk in wiki_chunks]

# 2. The model scores how well each chunk answers the claim
# Higher score = More relevant
scores = reranker.predict(pairs)

# 3. Zip the scores with the text and sort them from highest to lowest
ranked_results = sorted(zip(scores, wiki_chunks), reverse=True)

print("--- RE-RANKED EVIDENCE ---")
for score, chunk in ranked_results:
    # We now know exactly which text is the most useful!
    print(f"Relevance Score: {score:.2f} | Evidence: {chunk[:80]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

--- RE-RANKED EVIDENCE ---
Relevance Score: 8.82 | Evidence: The Moon is made of green cheese is a proverb and statement that refers to a fan...
Relevance Score: -8.13 | Evidence: The Moon is Earth's only natural satellite. It is a planetary-mass object with a...
Relevance Score: -9.52 | Evidence: Cheese is a dairy product produced in wide ranges of flavors, textures, and form...


In [5]:
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Load the NLI model (This will download for free and run on your CPU/GPU)
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

def verify_claim(claim, evidence):
    # NLI models expect a list of [Premise, Hypothesis]
    # Premise = Wikipedia Evidence | Hypothesis = Suspicious Claim
    prediction = nli_model.predict([evidence, claim])
    
    # The model returns 3 logits. We use softmax to turn them into probabilities (0 to 1)
    # The order for this model is [0: contradiction, 1: entailment, 2: neutral]
    exp_scores = np.exp(prediction)
    probabilities = exp_scores / np.sum(exp_scores)
    
    labels = ["REFUTED (Fake)", "SUPPORTED (True)", "NOT ENOUGH INFO"]
    result_idx = np.argmax(probabilities)
    
    return {
        "label": labels[result_idx],
        "confidence": probabilities[result_idx],
        "all_scores": dict(zip(labels, probabilities))
    }

# --- TEST IT OUT ---
test_claim = "The moon is made of green cheese."
test_evidence = "The Moon is made of green cheese is a proverb that refers to a fanciful belief."

final_verdict = verify_claim(test_claim, test_evidence)

print(f"Claim: {test_claim}")
print(f"Verdict: {final_verdict['label']}")
print(f"Confidence: {final_verdict['confidence']:.2%}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Claim: The moon is made of green cheese.
Verdict: REFUTED (Fake)
Confidence: 54.64%


In [ ]:
def fake_news_detector_pro(claim):
    # STEP 1: Search Wikipedia
    wikipedia.set_user_agent("research-assistant/0.1 kononov2005.d@gmail.com")
    raw_evidence = retrieve_wikipedia_evidence(claim, num_results=5)
    
    # STEP 2: Re-rank to find the best paragraph
    # (Assuming you use the reranker code from the previous step)
    pairs = [[claim, chunk] for chunk in raw_evidence]
    relevance_scores = reranker.predict(pairs)
    best_idx = np.argmax(relevance_scores)
    best_evidence = raw_evidence[best_idx]
    
    # Check if even the 'best' evidence is actually relevant
    if relevance_scores[best_idx] < 0:
        return "UNVERIFIED: No relevant information found on Wikipedia."

    # STEP 3: NLI Verification
    verdict = verify_claim(claim, best_evidence)
    
    return {
        "claim": claim,
        "verdict": verdict['label'],
        "confidence": f"{verdict['confidence']:.2%}",
        "source_text": best_evidence[:200] + "..."
    }

# --- THE FINAL DEMO ---
print(fake_news_detector_pro("Donald Trump was the 45th president of the United States"))
print(fake_news_detector_pro("The Earth is flat and supported by four elephants"))

Searching Wikipedia for: 'Donald Trump was the 45th president of the United States'...
Skipping 'Barron Trump' (Page not found)
{'claim': 'Donald Trump was the 45th president of the United States', 'verdict': 'SUPPORTED (True)', 'confidence': '99.44%', 'source_text': 'Donald John Trump (born June 14, 1946) is an American politician, media personality, and businessman who is the 47th president of the United States. A member of the Republican Party, he served as the ...'}
Searching Wikipedia for: 'The Earth is flat and supported by four elephants'...


In [1]:
import numpy as np

def calculate_weighted_pca_score(embeddings, weights):
    """
    embeddings: np.array of shape (n_docs, d_dimensions)
    weights: np.array of shape (n_docs,) - relevance scores
    """
    # 1. Softmax to normalize weights to sum to 1
    w = np.exp(weights) / np.sum(np.exp(weights))
    w = w.reshape(-1, 1) # Reshape for broadcasting

    # 2. Weighted Mean
    weighted_mean = np.sum(w * embeddings, axis=0)

    # 3. Center the data
    centered_data = embeddings - weighted_mean

    # 4. Weighted Covariance Matrix
    # We use the dot product of (X^T * W) and X
    weighted_covariance = (centered_data.T * w.T) @ centered_data

    # 5. Get Eigenvalues
    # eigvals_only is faster since we don't need the vectors yet
    eigenvalues = np.linalg.eigvalsh(weighted_covariance)
    
    # Sort descending
    eigenvalues = eigenvalues[::-1]

    # 6. Normalized Controversy Score
    # Ratio of variance in PC1 to total variance
    total_variance = np.sum(eigenvalues)
    if total_variance == 0: return 0
    
    controversy_score = eigenvalues[0] / total_variance
    
    return float(controversy_score)

# Example Usage:
# docs = 20 vectors, 384 dims
# relevance = 20 scores from Cross-Encoder
# score = calculate_weighted_pca_score(docs, relevance)

In [10]:
def calculate_centroid_consensus(embeddings, weights):
    # 1. Softmax to normalize weights
    w = np.exp(weights - np.max(weights))
    w = w / np.sum(w)
    w = w.reshape(-1, 1)

    # 2. L2 Normalize the embeddings (Crucial for Cosine Similarity)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norm_embeddings = embeddings / norms

    # 3. Calculate the Weighted Centroid (The 'Center of Gravity')
    centroid = np.sum(w * norm_embeddings, axis=0, keepdims=True)
    
    # 4. Normalize the Centroid itself
    centroid_norm = centroid / np.linalg.norm(centroid)

    # 5. Calculate Cosine Similarity of every document to the Centroid
    # Since vectors are normalized, dot product = cosine similarity
    similarities = np.dot(norm_embeddings, centroid_norm.T)

    # 6. Return the weighted average of these similarities
    consensus_score = np.sum(w * similarities)
    
    return float(consensus_score)

In [7]:
!pip install requests-cache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00


In [8]:
import requests_cache
requests_cache.install_cache('wiki_cache', expire_after=86400)

In [11]:
import numpy as np
import wikipedia
from sentence_transformers import SentenceTransformer, CrossEncoder

# ==========================================
# 1. Setup & Load Models
# ==========================================
# Set User-Agent to bypass Wikipedia's bot blocker!
wikipedia.set_user_agent("StudentProject_FakeNewsDetector/1.0 kononov2005.d@gmail.com")
# wikipedia.set_user_agent("StudentProject_FakeNewsDetector/1.0 (contact: yourname@example.com)")

print("Loading NLP models...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')          
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2') 

# ==========================================
# 2. Wikipedia Retrieval Function
# ==========================================
def retrieve_wikipedia_evidence(claim, num_results=15):
    print(f"\n🔍 Searching Wikipedia for: '{claim}' (fetching up to {num_results} pages)...")
    
    # 1. Get page titles
    search_results = wikipedia.search(claim, results=num_results)
    if not search_results:
        return []

    evidence_chunks = []
    
    # 2. Download page summaries
    for title in search_results:
        try:
            # We use auto_suggest=False to prevent Wiki from changing our search intent
            summary = wikipedia.summary(title, sentences=3, auto_suggest=False)
            evidence_chunks.append(summary)
        except wikipedia.exceptions.DisambiguationError:
            continue # Skip disambiguation pages
        except wikipedia.exceptions.PageError:
            continue # Skip missing pages
        except Exception:
            continue # Catch any weird API timeouts
            
    print(f"✅ Successfully retrieved {len(evidence_chunks)} Wikipedia summaries.")
    return evidence_chunks

# ==========================================
# 4. Master Evaluation Function
# ==========================================
def evaluate_claim(claim):
    # Step 1: Retrieve Live Evidence
    evidence = retrieve_wikipedia_evidence(claim, num_results=15)
    
    if len(evidence) < 3:
        return "🛑 ERROR: Not enough evidence found to analyze."

    # Step 2: Embed and Score
    vectors = embedder.encode(evidence)
    pairs = [[claim, doc] for doc in evidence]
    weights = reranker.predict(pairs)
    
    # Step 3: Run the Math
    pr_score = calculate_centroid_consensus(vectors, weights)
    
    # Step 4: The Verdict
    print(f"📊 Participation Ratio (Dimensions spanned): {pr_score:.2f}")
    
    # With 15 documents, a PR < 2.0 means severe polarization/controversy
    if pr_score < 2.0:
        print("🛑 VERDICT: CONTROVERSIAL / POLARIZED")
        print("The retrieved evidence splits into conflicting camps.")
    else:
        print("✅ VERDICT: CONSENSUS")
        print("The evidence forms a unified cluster. (Ready for NLI Fact-Check)")

# ==========================================
# 5. RUN THE TESTS
# ==========================================
evaluate_claim("Water boils at 100 degrees Celsius at sea level.")
print("-" * 50)
evaluate_claim("Universal Basic Income is highly beneficial for a nation's economy.")

Loading NLP models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 Searching Wikipedia for: 'Water boils at 100 degrees Celsius at sea level.' (fetching up to 15 pages)...
✅ Successfully retrieved 15 Wikipedia summaries.
📊 Participation Ratio (Dimensions spanned): 0.92
🛑 VERDICT: CONTROVERSIAL / POLARIZED
The retrieved evidence splits into conflicting camps.
--------------------------------------------------

🔍 Searching Wikipedia for: 'Universal Basic Income is highly beneficial for a nation's economy.' (fetching up to 15 pages)...
✅ Successfully retrieved 15 Wikipedia summaries.
📊 Participation Ratio (Dimensions spanned): 0.58
🛑 VERDICT: CONTROVERSIAL / POLARIZED
The retrieved evidence splits into conflicting camps.


In [13]:
import os
import urllib.request
import zipfile
import pandas as pd

import numpy as np
import requests_cache
import wikipedia
from datasets import load_dataset
from sentence_transformers import CrossEncoder

# 1. Setup Cache and NLI Model
requests_cache.install_cache('wiki_liar_cache', expire_after=86400)
wikipedia.set_user_agent("StudentProject_FakeNewsDetector/1.0 (contact: yourname@example.com)")

print("Loading DeBERTa NLI Model...")
# This model outputs 3 logits: [Contradiction, Entailment, Neutral]
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

# 2. Retrieval Function
def retrieve_top_k_wikipedia(claim, k=3):
    search_results = wikipedia.search(claim, results=k)
    evidence = []
    for title in search_results:
        try:
            summary = wikipedia.summary(title, sentences=3, auto_suggest=False)
            evidence.append(summary)
        except:
            continue
    return evidence

# 3. The Direct NLI Verifier
def verify_claim_nli(claim, evidence_list):
    if not evidence_list:
        return "UNVERIFIED"

    # Pair the claim with every piece of evidence: [[Premise, Hypothesis], ...]
    pairs = [[doc, claim] for doc in evidence_list]
    predictions = nli_model.predict(pairs)
    
    # Convert logits to probabilities using Softmax
    exp_scores = np.exp(predictions)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # We want to find the highest score for either Contradiction (index 0) or Entailment (index 1)
    best_contradiction = np.max(probs[:, 0])
    best_entailment = np.max(probs[:, 1])
    
    # Threshold for making a decision (e.g., must be > 60% confident)
    THRESHOLD = 0.60
    
    if best_entailment > best_contradiction and best_entailment > THRESHOLD:
        return "TRUE"
    elif best_contradiction > best_entailment and best_contradiction > THRESHOLD:
        return "FAKE"
    else:
        return "UNVERIFIED"


# 4. Dataset Mapping and Evaluation (Manual Pandas Version)
def evaluate_on_liar_manual(num_samples=50):
    print("1. Downloading raw LIAR dataset from UC Santa Barbara...")
    url = "http://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
    
    # Only download if we haven't already
    if not os.path.exists("liar_dataset.zip"):
        urllib.request.urlretrieve(url, "liar_dataset.zip")
    
    print("2. Extracting files...")
    with zipfile.ZipFile("liar_dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("liar_data")
    
    print("3. Loading and processing data with Pandas...")
    # Read the TSV. Missing closing parenthesis fixed!
    liar_test = pd.read_csv("liar_data/test.tsv", sep='\t', header=None, on_bad_lines='skip')
    
    # RAW TSV MAPPING: The raw file uses strings in Column 1, not integers.
    # Column 1 = label, Column 2 = statement/claim
    label_map = {
        "pants-fire": "FAKE", 
        "false": "FAKE", 
        "barely-true": "FAKE", 
        "mostly-true": "TRUE", 
        "true": "TRUE"
    }

    correct = 0
    total_verifiable = 0
    unverified = 0
    processed_count = 0
    
    print("\n🚀 Beginning Evaluation Pipeline...\n")

    # Iterate row by row using Pandas
    for index, row in liar_test.iterrows():
        # Stop once we hit the requested number of samples
        if processed_count >= num_samples:
            break
            
        raw_label = str(row[1]).lower()
        claim = str(row[2])
        
        # Skip the ambiguous 'half-true' label or any malformed rows
        if raw_label == "half-true" or raw_label not in label_map: 
            continue 
            
        expected_label = label_map[raw_label]
        
        # --- RUN THE PIPELINE ---
        # (Make sure retrieve_top_k_wikipedia and verify_claim_nli are defined above this)
        evidence = retrieve_top_k_wikipedia(claim, k=3)
        predicted_label = verify_claim_nli(claim, evidence)
        
        print(f"[{processed_count+1}/{num_samples}] Claim: {claim[:80]}...")
        print(f"Expected: {expected_label} | Predicted: {predicted_label}\n")
        
        # --- TALLY METRICS ---
        if predicted_label == "UNVERIFIED":
            unverified += 1
        else:
            total_verifiable += 1
            if predicted_label == expected_label:
                correct += 1
                
        processed_count += 1

    # Print Final Metrics
    print("="*40)
    print("📊 LIAR DATASET RESULTS (ZERO-SHOT RAG)")
    print("="*40)
    print(f"Total Claims Tested: {processed_count}")
    print(f"Unverified (No Wikipedia Data): {unverified}")
    print(f"Successfully Verified: {total_verifiable}")
    
    if total_verifiable > 0:
        accuracy = correct / total_verifiable
        print(f"Accuracy on Verified Claims: {accuracy:.2%}")
    else:
        print("Accuracy: N/A (Could not verify any claims)")

# Run the test
evaluate_on_liar_manual(num_samples=20)

1. Downloading raw LIAR dataset from UC Santa Barbara...
2. Extracting files...
3. Loading and processing data with Pandas...

🚀 Beginning Evaluation Pipeline...

[1/20] Claim: Building a wall on the U.S.-Mexico border will take literally years....
Expected: TRUE | Predicted: UNVERIFIED

[2/20] Claim: Wisconsin is on pace to double the number of layoffs this year....
Expected: FAKE | Predicted: UNVERIFIED

[3/20] Claim: Says John McCain has done nothing to help the vets....
Expected: FAKE | Predicted: UNVERIFIED

[4/20] Claim: When asked by a reporter whether hes at the center of a criminal scheme to viola...
Expected: FAKE | Predicted: FAKE

[5/20] Claim: Over the past five years the federal government has paid out $601 million in ret...
Expected: TRUE | Predicted: UNVERIFIED

[6/20] Claim: Says that Tennessee law requires that schools receive half of proceeds -- $31 mi...
Expected: TRUE | Predicted: FAKE

[7/20] Claim: Says Vice President Joe Biden "admits that the American people ar

In [220]:
print(liar_train.head(1))

           0      1                                                  2  \
0  2635.json  false  Says the Annies List political group supports ...   

          3             4                     5      6           7    8    9  \
0  abortion  dwayne-bohac  State representative  Texas  republican  0.0  1.0   

    10   11   12        13 clean_label  \
0  0.0  0.0  0.0  a mailer       false   

                                           statement  binary_label  
0  Says the Annies List political group supports ...             0  


In [222]:
liar_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10240 entries, 0 to 10239
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   0             10240 non-null  object 
 1   1             10240 non-null  object 
 2   2             10240 non-null  object 
 3   3             10238 non-null  object 
 4   4             10238 non-null  object 
 5   5             7342 non-null   object 
 6   6             8030 non-null   object 
 7   7             10238 non-null  object 
 8   8             10238 non-null  float64
 9   9             10238 non-null  float64
 10  10            10238 non-null  float64
 11  11            10238 non-null  float64
 12  12            10238 non-null  float64
 13  13            10138 non-null  object 
 14  clean_label   10240 non-null  object 
 15  statement     7342 non-null   object 
 16  binary_label  10240 non-null  int64  
dtypes: float64(5), int64(1), object(11)
memory usage: 1.3+ MB


In [19]:
tokenizer = GloveTokenizer(texts_train)

In [259]:
import pandas as pd
import urllib.request
import zipfile
import os
import torch
from torch.utils.data import DataLoader

print("1. Downloading raw LIAR dataset from UC Santa Barbara...")
url = "http://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
urllib.request.urlretrieve(url, "liar_dataset.zip")

print("2. Extracting files...")
with zipfile.ZipFile("liar_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("liar_data")

print("3. Loading and processing data with Pandas...")
# Safest way to read the TSV: don't guess column names, just use indices
liar_train = pd.read_csv("liar_data/train.tsv", sep='\t', header=None, on_bad_lines='skip')
liar_val = pd.read_csv("liar_data/valid.tsv", sep='\t', header=None, on_bad_lines='skip')
liar_test = pd.read_csv("liar_data/test.tsv", sep='\t', header=None, on_bad_lines='skip')

def get_texts_and_labels(liar_df, binary=True):
    liar_df['clean_label'] = liar_df[1].astype(str).str.lower().str.strip()
    # liar_df['statement'] = liar_df[2].astype(str)
    liar_df['statement']='Subject: '+liar_df[3].fillna('')+'. '+liar_df[5].fillna('')+' '+liar_df[7].fillna('')+'. '+liar_df[2]
    liar_df['statement'] = liar_df['statement'].astype(str).str.lower().str.strip()
    # Map the 6-way labels to Binary
    if binary:
        label_mapping = {
            'pants-fire': 0,
            'false': 0,
            'barely-true': 0,
            'half-true': 1,
            'mostly-true': 1,
            'true': 1
        }
    else:
        label_mapping = {
            'pants-fire': 0,
            'false': 1,
            'barely-true': 2,
            'half-true': 3,
            'mostly-true': 4,
            'true': 5
        }
    
    liar_df['binary_label'] = liar_df['clean_label'].map(label_mapping)
    
    # Drop any rows that STILL failed the mapping (just in case)
    liar_df = liar_df.dropna(subset=['binary_label'])
    
    # Extract standard Python lists
    liar_texts = liar_df['statement'].tolist()
    liar_texts = [preprocess_text(text) for text in liar_texts]
    liar_labels = liar_df['binary_label'].astype(int).tolist()
    
    print(f"Successfully loaded {len(liar_texts)} unseen statements.")
    
    # Crash prevention check:
    if len(liar_texts) == 0:
        raise ValueError("CRITICAL: No data was loaded! The label mapping failed.")
    return liar_texts, liar_labels


texts_train, labels_train = get_texts_and_labels(liar_train, binary=True)
texts_val, labels_val = get_texts_and_labels(liar_val, binary=True)
texts_test, labels_test = get_texts_and_labels(liar_test, binary=True)

tokenizer = GloveTokenizer(texts_train)
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("roberta-base")

batch_size = 32
max_len = 200

train_ds = FakeNewsDataset(
    raw_texts=texts_train, 
    labels=labels_train, 
    tokenizer=tokenizer, 
    max_seq_len=max_len
)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

val_ds = FakeNewsDataset(
    raw_texts=texts_val, 
    labels=labels_val, 
    tokenizer=tokenizer, 
    max_seq_len=max_len
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=True)

test_ds = FakeNewsDataset(
    raw_texts=texts_test, 
    labels=labels_test, 
    tokenizer=tokenizer, 
    max_seq_len=max_len 
)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

1. Downloading raw LIAR dataset from UC Santa Barbara...
2. Extracting files...
3. Loading and processing data with Pandas...
Successfully loaded 10240 unseen statements.
Successfully loaded 1284 unseen statements.
Successfully loaded 1267 unseen statements.


In [29]:
num_epochs = 3
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs)

Epoch 1/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 379.43it/s, val_loss=0.6980]


Train loss: 0.689, Train acc: 0.446
Val loss: 0.694, Val acc: 0.48


Epoch 2/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 382.97it/s, val_loss=0.6390]


Train loss: 0.686, Train acc: 0.439
Val loss: 0.695, Val acc: 0.48


Epoch 3/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 386.46it/s, val_loss=0.7082]

Train loss: 0.686, Train acc: 0.438
Val loss: 0.701, Val acc: 0.48


In [56]:
eval_model(model, test_loader)

Epoch 1/1 [Valid]: 100%|██████████| 40/40 [00:00<00:00, 306.99it/s, val_loss=0.6875]

Val acc:  0.4388318863456985


## Performance of random forest on binary labels

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Vectorization: Convert text to numbers
# We limit max_features to 5000 to keep the forest fast and avoid memory issues
vectorizer = TfidfVectorizer(stop_words='english', tokenizer=None, 
                             ngram_range=(1,1), max_features=5000)

print("Vectorizing text...")

# X_train = tokenizer(
#     text = texts_train,
#     padding = 'max_length',
#     truncation = True,
#     max_length = 100,
#     return_tensors = 'np'
# )['input_ids']
# X_test = tokenizer(
#     text = texts_test,
#     padding = 'max_length',
#     truncation = True,
#     max_length = 100,
#     return_tensors = 'np'
# )['input_ids']

# print("Vectorizing text...")

X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_val)

# 2. Initialize Random Forest
# n_jobs=-1 uses all available CPU cores
# n_estimators=100 is a good balance between speed and accuracy
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

print("Training Random Forest (this might take a minute)...")
rf_model.fit(X_train, labels_train)

# 3. Predict and Evaluate
print("Evaluating...")
y_pred = rf_model.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(labels_val, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(labels_val, y_pred))

Vectorizing text...
Training Random Forest (this might take a minute)...
Evaluating...
Random Forest Accuracy: 0.5966

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.43      0.51       616
           1       0.59      0.75      0.66       668

    accuracy                           0.60      1284
   macro avg       0.60      0.59      0.58      1284
weighted avg       0.60      0.60      0.59      1284



## Performance of random forest on multi class labels

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Vectorization: Convert text to numbers
# We limit max_features to 5000 to keep the forest fast and avoid memory issues
vectorizer = TfidfVectorizer(stop_words='english', tokenizer=None, 
                             ngram_range=(1,1), max_features=5000)

print("Vectorizing text...")

# X_train = tokenizer(
#     text = texts_train,
#     padding = 'max_length',
#     truncation = True,
#     max_length = 100,
#     return_tensors = 'np'
# )['input_ids']
# X_test = tokenizer(
#     text = texts_test,
#     padding = 'max_length',
#     truncation = True,
#     max_length = 100,
#     return_tensors = 'np'
# )['input_ids']

# print("Vectorizing text...")

X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_val)

# 2. Initialize Random Forest
# n_jobs=-1 uses all available CPU cores
# n_estimators=100 is a good balance between speed and accuracy
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, min_samples_split=5, random_state=42)

print("Training Random Forest (this might take a minute)...")
rf_model.fit(X_train, labels_train)

# 3. Predict and Evaluate
print("Evaluating...")
y_pred = rf_model.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(labels_val, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(labels_val, y_pred))

Vectorizing text...
Training Random Forest (this might take a minute)...
Evaluating...
Random Forest Accuracy: 0.2539

Classification Report:
              precision    recall  f1-score   support

 barely-true       0.20      0.14      0.16       237
       false       0.27      0.39      0.32       263
   half-true       0.24      0.28      0.26       248
 mostly-true       0.27      0.32      0.29       251
  pants-fire       0.41      0.09      0.15       116
        true       0.22      0.18      0.20       169

    accuracy                           0.25      1284
   macro avg       0.27      0.23      0.23      1284
weighted avg       0.26      0.25      0.24      1284



In [260]:
NUM_LAYERS = 1
VOCAB_SIZE = tokenizer.vocab_size
# lstm_best_model_liar_cpu.pt (gpu tensors, bug of name) is for hidden_dim=64, embed=100
HIDDEN_DIM = 128
EMBEDDING_DIM = 100
OUTPUT_DIM = 1
DROP_PROB = 0.25
model = FakeNewsLstm(no_layers=NUM_LAYERS, vocab_size=VOCAB_SIZE,
                    hidden_dim=HIDDEN_DIM, embedding_dim=EMBEDDING_DIM,
                    output_dim=OUTPUT_DIM, drop_prob=DROP_PROB).to(device)

In [101]:
!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-05-09 10:24:50--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.08MB/s    in 2m 55s  

2026-05-09 10:27:46 (4.71 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [261]:
import numpy as np

def create_glove_matrix(glove_file_path, word_index, embed_dim=100):
    vocab_size = len(word_index) 
    # Initialize exactly 10,002 rows
    pretrained_weights = np.zeros((vocab_size, embed_dim), dtype=np.float32)
    words_found = 0
    
    print("Parsing GloVe file...")
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            
            # If the GloVe word is in our 10,002 tokenizer vocab...
            if word in word_index:
                idx = word_index[word]
                vector = np.array(values[1:], dtype=np.float32)
                pretrained_weights[idx] = vector
                words_found += 1
                
    print(f"Success: Found GloVe vectors for {words_found} out of {vocab_size} words.")
    return pretrained_weights

# --- Generate the Weights ---
# (Assuming your GloveTokenizer object is named 'tokenizer')
glove_weights = create_glove_matrix('glove.6B.100d.txt', tokenizer.word_index, embed_dim=100)

Parsing GloVe file...
Success: Found GloVe vectors for 8035 out of 10002 words.


In [95]:
from transformers import RobertaModel
from sklearn.decomposition import PCA

base_model = RobertaModel.from_pretrained('roberta-base')

# 2. Extract the embedding matrix as a numpy array
# This will have a shape of (50265, 768)
pretrained_weights = base_model.embeddings.word_embeddings.weight.detach().cpu().numpy()

print(f"Pretrained weights captured! Shape: {pretrained_weights.shape}")
pca = PCA(n_components=100)
compressed_weights = pca.fit_transform(pretrained_weights)

print(f"New compressed shape: {compressed_weights.shape}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Pretrained weights captured! Shape: (50265, 768)
New compressed shape: (50265, 100)


In [262]:
# self.embedding = nn.Embedding(vocab_size, 100)
# 4. Copy the weights and allow the model to fine-tune them during training
model.embedding.weight.data.copy_(torch.from_numpy(glove_weights))
model.embedding.weight.requires_grad = False
# model.embedding.weight.requires_grad = True

In [263]:
optimizer = Adam(model.parameters(), lr=2e-4)
sched = StepLR(optimizer, step_size=30, gamma=0.1)
criterion = nn.BCEWithLogitsLoss()

In [271]:
num_epochs = 3
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True, savepath='lstm128_8k.pt')

Epoch 1/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 238.34it/s, val_loss=1.3635]


Train loss: 0.525, Train acc: 0.739
Val loss: 0.721, Val acc: 0.624


Epoch 2/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 258.08it/s, val_loss=0.9046]


Train loss: 0.509, Train acc: 0.75
Val loss: 0.713, Val acc: 0.612


Epoch 3/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 253.96it/s, val_loss=0.9864]

Train loss: 0.498, Train acc: 0.755
Val loss: 0.759, Val acc: 0.605


## Training with pretrained embeddings for Roberta-Base

In [110]:
num_epochs = 3
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=False)

Epoch 1/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 336.36it/s, val_loss=1.8851]


Train loss: 1.762, Train acc: 0.2
Val loss: 1.764, Val acc: 0.193


Epoch 2/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 326.82it/s, val_loss=1.9959]


Train loss: 1.759, Train acc: 0.203
Val loss: 1.774, Val acc: 0.193


Epoch 3/3 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 285.39it/s, val_loss=1.8520]

Train loss: 1.759, Train acc: 0.204
Val loss: 1.765, Val acc: 0.193


In [102]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True)

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 313.38it/s, val_loss=0.5378]


Train loss: 0.618, Train acc: 0.674
Val loss: 0.653, Val acc: 0.619


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 312.41it/s, val_loss=0.5192]


Train loss: 0.616, Train acc: 0.674
Val loss: 0.653, Val acc: 0.621


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 329.22it/s, val_loss=0.5990]


Train loss: 0.614, Train acc: 0.677
Val loss: 0.654, Val acc: 0.623


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 317.09it/s, val_loss=0.5304]


Train loss: 0.613, Train acc: 0.678
Val loss: 0.653, Val acc: 0.62


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 327.58it/s, val_loss=0.6393]

Train loss: 0.612, Train acc: 0.677
Val loss: 0.656, Val acc: 0.619


In [103]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True)

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 329.17it/s, val_loss=0.6580]


Train loss: 0.612, Train acc: 0.682
Val loss: 0.656, Val acc: 0.619


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 333.97it/s, val_loss=0.7545]


Train loss: 0.612, Train acc: 0.68
Val loss: 0.658, Val acc: 0.621


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 315.45it/s, val_loss=0.6826]


Train loss: 0.611, Train acc: 0.68
Val loss: 0.657, Val acc: 0.621


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 323.80it/s, val_loss=0.4774]


Train loss: 0.61, Train acc: 0.68
Val loss: 0.652, Val acc: 0.62


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 326.47it/s, val_loss=0.4974]


Train loss: 0.611, Train acc: 0.679
Val loss: 0.653, Val acc: 0.621


In [118]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True)

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 296.11it/s, val_loss=0.7348]


Train loss: 0.678, Train acc: 0.565
Val loss: 0.674, Val acc: 0.57


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 298.29it/s, val_loss=0.8134]


Train loss: 0.663, Train acc: 0.595
Val loss: 0.671, Val acc: 0.59


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 296.97it/s, val_loss=0.5094]


Train loss: 0.653, Train acc: 0.615
Val loss: 0.659, Val acc: 0.618


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 298.63it/s, val_loss=1.1462]


Train loss: 0.641, Train acc: 0.635
Val loss: 0.668, Val acc: 0.615


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 297.43it/s, val_loss=1.2311]

Train loss: 0.613, Train acc: 0.675
Val loss: 0.681, Val acc: 0.625


In [250]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True, 
    savepath='/kaggle/working/lstm128_full_liar_gpu.pt')

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 256.48it/s, val_loss=0.7897]


Train loss: 0.559, Train acc: 0.707
Val loss: 0.668, Val acc: 0.624


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 256.73it/s, val_loss=0.4504]


Train loss: 0.539, Train acc: 0.725
Val loss: 0.684, Val acc: 0.627


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 262.00it/s, val_loss=0.3541]


Train loss: 0.519, Train acc: 0.738
Val loss: 0.687, Val acc: 0.613


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 260.32it/s, val_loss=0.3625]


Train loss: 0.51, Train acc: 0.743
Val loss: 0.712, Val acc: 0.608


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 263.47it/s, val_loss=0.5484]


Train loss: 0.5, Train acc: 0.749
Val loss: 0.728, Val acc: 0.6


In [120]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embedding_dim]
        x = x + self.pe[:, :x.size(1), :]
        return x

class FakeNewsTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, num_heads=4, hidden_dim=128, num_layers=1, output_dim=1, drop_prob=0.35):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoder = PositionalEncoding(embedding_dim)
        
        # Note: d_model MUST equal embedding_dim. num_heads must evenly divide d_model (100 / 4 = 25)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, 
            nhead=num_heads, 
            dim_feedforward=hidden_dim, 
            dropout=drop_prob, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def init_hidden(self, batch_size, device):
        return None

    def forward(self, x, hidden=None):
        # x shape: [batch_size, seq_len]
        
        # 1. Create a mask so the Transformer ignores <PAD> tokens (assumes PAD ID is 0)
        pad_mask = (x == 0)
        
        embeds = self.embedding(x)
        embeds = self.pos_encoder(embeds)
        
        # Output shape: [batch_size, seq_len, embedding_dim]
        transformer_out = self.transformer_encoder(embeds, src_key_padding_mask=pad_mask)
        
        # Max pooling finds the strongest signals (most important words) across the sequence.
        pooled_out, _ = transformer_out.max(dim=1)
        
        out = self.dropout(pooled_out)
        out = self.fc(out)
        
        return out, hidden

In [175]:
# The setup
model = FakeNewsTransformer(
    vocab_size=VOCAB_SIZE,
    embedding_dim=100,  # Must be 100 to match GloVe
    num_heads=4,        # 4 heads works well because 100 / 4 is a whole number
    hidden_dim=50,     # The internal feed-forward network size
    num_layers=1,       # Keep at 1 to prevent overfitting
    output_dim=1,
    drop_prob=0.35
).to(device)

# Load your GloVe weights!
model.embedding.weight.data.copy_(torch.from_numpy(glove_weights))
model.embedding.weight.requires_grad = False # Keep them frozen!

In [176]:
optimizer = Adam(model.parameters(), lr=1e-3)
sched = StepLR(optimizer, step_size=30, gamma=0.1)
criterion = nn.BCEWithLogitsLoss()

## Train Transformer model

In [246]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True, 
    savepath='/kaggle/working/lstm128_full_liar_gpu.pt')

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 263.56it/s, val_loss=0.5003]


Train loss: 0.619, Train acc: 0.661
Val loss: 0.657, Val acc: 0.623


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 259.67it/s, val_loss=0.6271]


Train loss: 0.613, Train acc: 0.67
Val loss: 0.645, Val acc: 0.628


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 264.47it/s, val_loss=1.0091]


Train loss: 0.607, Train acc: 0.674
Val loss: 0.656, Val acc: 0.631


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 263.00it/s, val_loss=0.8578]


Train loss: 0.6, Train acc: 0.679
Val loss: 0.669, Val acc: 0.615


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 224.34it/s, val_loss=0.4384]


Train loss: 0.596, Train acc: 0.683
Val loss: 0.646, Val acc: 0.636


In [267]:
model.load_state_dict(torch.load('/kaggle/working/top_model.pt', weights_only=True))
eval_model(model, val_loader)

Epoch 1/1 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 200.06it/s, val_loss=0.3045]


Val acc:  0.6401869158878505


In [268]:
eval_model(model, test_loader)

Epoch 1/1 [Valid]: 100%|██████████| 40/40 [00:00<00:00, 201.32it/s, val_loss=0.8314]

Val acc:  0.6258879242304657


In [256]:
print(tokenizer('fake news detection project on liar dataset with lstm model'))

{'input_ids': [0, 18169, 340, 12673, 695, 15, 28587, 41616, 19, 784, 620, 119, 1421, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [248]:
eval_model(model, test_loader)

Epoch 1/1 [Valid]: 100%|██████████| 40/40 [00:00<00:00, 242.01it/s, val_loss=0.8242]

Val acc:  0.6179952644041041


In [257]:
num_epochs = 5
train_loss_list, val_loss_list, train_acc_list, val_acc_list = train_epochs(
    model, optimizer, criterion, scheduler=sched, epochs=num_epochs, binary=True, 
    savepath='/kaggle/working/lstm128_full_liar_gpu.pt')

Epoch 1/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 236.12it/s, val_loss=0.7675]


Train loss: 0.686, Train acc: 0.562
Val loss: 0.698, Val acc: 0.52


Epoch 2/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 236.80it/s, val_loss=0.6981]


Train loss: 0.687, Train acc: 0.561
Val loss: 0.694, Val acc: 0.52


Epoch 3/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 209.94it/s, val_loss=0.6627]


Train loss: 0.687, Train acc: 0.562
Val loss: 0.692, Val acc: 0.52


Epoch 4/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 216.42it/s, val_loss=0.5817]


Train loss: 0.687, Train acc: 0.56
Val loss: 0.693, Val acc: 0.52


Epoch 5/5 [Valid]: 100%|██████████| 41/41 [00:00<00:00, 228.68it/s, val_loss=0.5804]

Train loss: 0.687, Train acc: 0.561
Val loss: 0.693, Val acc: 0.52
